# LoRA Trainer v2.1 (SDXL + `jobs/<job_id>`)

Pinalitan ang SD 1.5 na `train_network.py` ng SDXL na `sdxl_train_network.py`, at
ang flat na `datasets/` + `output/` ng `jobs/<job_id>/` protocol ng app. Doon
lang magiging end-to-end: nababasa ng `src/direk/colabTrainer.ts` ang
`status.json` at `manifest.json`, at maipapasa ang `.safetensors` sa ComfyUI.

Dalawa pa ring mode:

| Mode | Para saan |
| --- | --- |
| **Job mode** (default) | `jobs/<job_id>/` — kumpleto ang `status.json` + `manifest.json`. Ito ang konektado sa app. |
| **Manual mode** | `datasets/` + `output/` — flat, para sa mabilisang pagsubok. Hindi ito makikita ng app. |

Ang job mode ay **hindi nagsisimula ng sariling training loop**. Sinusunod nito
ang `job.json` na isinulat ng app: `train_args`, `dataset`, `output`,
`base_model`, `output_name`. Isang source of truth lang.


## Cell 1 - GPU check


In [ ]:
!nvidia-smi


## Cell 2 - Mount Drive at piliin ang mode


In [ ]:
from google.colab import drive
import os, glob, json, time, hashlib

drive.mount("/content/drive")

DRIVE_ROOT = os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive/LORA_TRAINING")
assert os.path.isdir(DRIVE_ROOT), f"Missing folder: {DRIVE_ROOT}"

# "job" = konektado sa app. "manual" = flat folders, para sa pagsubok lang.
MODE = os.environ.get("MODE", "job")
JOB_ID = os.environ.get("JOB_ID", "")  # blangko = pinakabagong QUEUED job

def job_dir(job_id):
    return os.path.join(DRIVE_ROOT, "jobs", job_id)

def read_json(path, default=None):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return default

def write_json(path, doc):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(doc, f, indent=2)
    os.replace(tmp, path)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

print("MODE:", MODE, "| DRIVE_ROOT:", DRIVE_ROOT)


## Cell 3 - Install sd-scripts

Ang `xformers` ay **hindi** kasama sa `requirements.txt` ng sd-scripts, at ang
`model_io.py` ay may `import xformers.ops` sa loob ng branch nito. Kaya hindi
ito gagamitin; `--sdpa` ang pipiliin sa config, at may built-in na SDPA ang
PyTorch 2.x.


In [ ]:
import subprocess, os

SD_SCRIPTS = "/content/sd-scripts"
if not os.path.isdir(SD_SCRIPTS):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/kohya-ss/sd-scripts.git", SD_SCRIPTS],
                   check=True)
else:
    subprocess.run(["git", "-C", SD_SCRIPTS, "pull", "-q"], check=False)

os.chdir(SD_SCRIPTS)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["pip", "install", "-q", "bitsandbytes"], check=True)

# Ang SDXL trainer mismo, hindi lang ang base trainer.
trainer = os.path.join(SD_SCRIPTS, "sdxl_train_network.py")
assert os.path.isfile(trainer), f"Wala ang {trainer} - SDXL trainer ang kailangan."
print("sd-scripts ready:", SD_SCRIPTS)
print("trainer:", trainer)


## Cell 4 - Piliin ang job, at basahin ang `job.json`

Kapag `MODE = "job"`, ang `job.json` ang nagtatakda ng lahat: base model,
dataset, output, at training args. Walang hardcoded na landas dito.


In [ ]:
def pick_job():
    if JOB_ID:
        return JOB_ID
    jobs = sorted(glob.glob(os.path.join(DRIVE_ROOT, "jobs", "*")))
    for d in reversed(jobs):
        st = read_json(os.path.join(d, "status.json"), {})
        if st.get("state", "QUEUED") == "QUEUED":
            return os.path.basename(d)
    return os.path.basename(jobs[-1]) if jobs else ""

job = None
if MODE == "job":
    job_id = pick_job()
    assert job_id, f"Walang job sa {DRIVE_ROOT}/jobs"
    job = read_json(os.path.join(job_dir(job_id), "job.json"))
    assert job, f"Hindi mabasa ang job.json para sa {job_id}"
    # Ang folder name ang may bisa, hindi ang field: isang pagkakamali sa
    # job_id ay magsusulat ng status sa maling job.
    assert job["job_id"] == job_id, (
        f"job.json ay para sa {job['job_id']}, pero ang folder ay {job_id}"
    )
    args = job["train_args"]
    dataset_dir = os.path.join(DRIVE_ROOT, job["dataset"])
    output_dir = os.path.join(DRIVE_ROOT, job["output"])
    base_model = job["base_model"]
    output_name = job["output_name"]
    os.makedirs(output_dir, exist_ok=True)
    print("job:", job_id)
    print(json.dumps(job, indent=2))
else:
    DATASETS = os.path.join(DRIVE_ROOT, "datasets")
    OUTPUT = os.path.join(DRIVE_ROOT, "output")
    os.makedirs(OUTPUT, exist_ok=True)
    dataset_dir = DATASETS
    output_dir = OUTPUT
    base_model = os.environ.get("BASE_MODEL", "")
    output_name = "manual"
    print("manual mode:", DATASETS, "->", OUTPUT)


## Cell 5 - Kunin ang base model, at tiyaking SDXL ito

Ang app ay SDXL ang default (`colabTrainer.ts:345`). Kapag SD 1.5 ang na-download,
hindi magagamit ng renderer ang resulta. Kaya may hard check dito.


In [ ]:
from huggingface_hub import snapshot_download

SDXL_DEFAULT = "stabilityai/stable-diffusion-xl-base-1.0"

if MODE == "job":
    # Isang HuggingFace id, o isang landas sa loob ng Drive.
    if "/" in base_model and not os.path.exists(base_model):
        local_base = snapshot_download(
            base_model,
            local_dir=job.get("colab_base_model_dir", "/content/models"),
        )
    else:
        local_base = base_model
else:
    local_base = base_model or SDXL_DEFAULT
    if "/" in local_base and not os.path.exists(local_base):
        local_base = snapshot_download(local_base, local_dir="/content/models")

# SDXL ay ~6.9 GB sa fp16 (unet 5.1 + vae 0.3 + text encoders 1.4).
# SD 1.5 ay ~4 GB. Kaya ang laki ay isang mabilis na pagkakakilanlan.
total_gb = 0.0
if os.path.isdir(local_base):
    for root, _, files in os.walk(local_base):
        for f in files:
            if f.endswith((".safetensors", ".bin", ".ckpt")):
                total_gb += os.path.getsize(os.path.join(root, f)) / (1024 ** 3)
else:
    total_gb = os.path.getsize(local_base) / (1024 ** 3)

print(f"base model: {local_base} ({total_gb:.2f} GB)")
assert total_gb > 5.0, (
    f"Masyadong maliit ({total_gb:.2f} GB) para sa SDXL. SD 1.5 ito, o kulang ang "
    f"download. Itakda ang KOHYA_BASE_MODEL={SDXL_DEFAULT} sa app, o ayusin ang "
    f"lokal na landas."
)
print("SDXL OK")


## Cell 6 - Dataset checker

Sa job mode, isang folder lang ang sinusuri: ang `dataset/` ng job. Sa manual
mode, lahat ng subfolder ng `datasets/`.


In [ ]:
IMG_EXT = (".png", ".jpg", ".jpeg", ".webp", ".bmp")

def check_folder(path, min_images):
    files = os.listdir(path)
    images = [f for f in files if f.lower().endswith(IMG_EXT)]
    captions = [f for f in files if f.lower().endswith(".txt")]
    missing = [f for f in images if not os.path.exists(
        os.path.join(path, os.path.splitext(f)[0] + ".txt"))]
    ok = len(images) >= min_images
    print(f"{os.path.basename(path)}/  images={len(images)} captions={len(captions)} "
          f"walang-caption={len(missing)} -> {'READY' if ok else 'SKIP (kulang)'}")
    if missing[:3]:
        print("   halimbawa ng walang caption:", missing[:3])
    return ok

MIN_IMAGES = 5

if MODE == "job":
    assert check_folder(dataset_dir, MIN_IMAGES), (
        f"Kulang sa {MIN_IMAGES} images ang {dataset_dir}"
    )
    READY = [dataset_dir]
else:
    READY = []
    for sub in sorted(os.listdir(dataset_dir)):
        p = os.path.join(dataset_dir, sub)
        if os.path.isdir(p) and check_folder(p, MIN_IMAGES):
            READY.append(p)
    assert READY, "Walang handang dataset. Mag-upload muna ng images sa Drive."

print(f"\nREADY: {len(READY)} folder")


## Cell 7 - Isulat ang kohya config, tapos mag-train

TOML ang gamit (`--config_file`) dahil iyon ang sinusuportahan ng
`sdxl_train_network.py` (`library/args.py:609`), at iyon din ang binabasa ng
`parseKohyaConfig()` sa app. Parehong config, parehong resulta.


In [ ]:
import sys

def write_config(path, dataset, out_dir, out_name, cfg):
    toml = f'''# Generated from job.json - SDXL
[general]
enable_bucket = true
bucket_no_upscale = false
output_dir = "{out_dir}"
output_name = "{out_name}"
save_model_as = "{cfg["save_model_as"]}"

[[datasets]]
resolution = {cfg["resolution"]}
batch_size = {cfg["batch_size"]}
enable_bucket = true

  [[datasets.subsets]]
  image_dir = "{dataset}"
  caption_extension = ".txt"
  num_repeats = {cfg["num_repeats"]}

[network_arguments]
network_module = "{cfg["network_module"]}"
network_dim = {cfg["network_dim"]}
network_alpha = {cfg["network_alpha"]}

[optimizer_arguments]
learning_rate = {cfg["learning_rate"]}
optimizer_type = "{cfg["optimizer_type"]}"
lr_scheduler = "{cfg["lr_scheduler"]}"

[training_arguments]
pretrained_model_name_or_path = "{local_base}"
max_train_epochs = {cfg["max_train_epochs"]}
save_every_n_epochs = {cfg["save_every_n_epochs"]}
mixed_precision = "{cfg["mixed_precision"]}"
seed = {cfg["seed"]}
gradient_checkpointing = true
sdpa = true
cache_latents = true

[logging_arguments]
logging_dir = "{out_dir}/logs"
'''
    with open(path, "w") as f:
        f.write(toml)
    return path

if MODE == "job":
    config_path = os.path.join(job_dir(job_id), "kohya_config.toml")
    write_config(config_path, dataset_dir, output_dir, output_name, args)
else:
    # Ang manual mode ay gumagamit ng katulad na default, pero 1024 para sa SDXL.
    manual_args = {
        "resolution": 1024, "batch_size": 1, "num_repeats": 10,
        "network_module": "networks.lora", "network_dim": 32, "network_alpha": 16,
        "learning_rate": 1e-4, "optimizer_type": "AdamW8bit",
        "lr_scheduler": "cosine", "max_train_epochs": 10,
        "save_every_n_epochs": 2, "mixed_precision": "fp16", "seed": 42,
        "save_model_as": "safetensors",
    }
    config_path = os.path.join(DRIVE_ROOT, "kohya_config_manual.toml")
    write_config(config_path, READY[0], output_dir, output_name, manual_args)

print(open(config_path).read())


## Cell 8 - Training (may `status.json` at `cancel.flag`)

Bawat 2 segundo, sinusuri ang `cancel.flag` para maputol nang malinis sa gitna
ng run. Kapag nag-crash, `FAILED` ang isusulat — hindi `DONE`.


In [ ]:
cancel_flag = os.path.join(job_dir(job_id), "cancel.flag") if MODE == "job" else ""

def set_status(state, **extra):
    if MODE != "job":
        return
    doc = {"state": state,
           "updated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    doc.update(extra)
    write_json(os.path.join(job_dir(job_id), "status.json"), doc)
    print("status ->", state, {k: v for k, v in extra.items() if k != "message"})

existing = glob.glob(os.path.join(output_dir, "*.safetensors"))
if existing and not (cancel_flag and os.path.exists(cancel_flag)):
    print("May umiiral nang output, laktawan ang training:", existing)
else:
    set_status("RUNNING", epoch=0, total_epochs=args["max_train_epochs"] if MODE == "job" else None)
    cmd = [sys.executable, os.path.join(SD_SCRIPTS, "sdxl_train_network.py"),
           "--config_file", config_path]
    print("$", " ".join(cmd))

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    epoch = 0
    last_check = time.time()
    for line in proc.stdout:
        print(line, end="")
        low = line.lower()
        if "epoch" in low and ("steps" in low or "/" in line):
            epoch += 1
            set_status("RUNNING", epoch=epoch,
                       total_epochs=args["max_train_epochs"] if MODE == "job" else None,
                       message=line.strip()[:200])
        if cancel_flag and time.time() - last_check > 2:
            last_check = time.time()
            if os.path.exists(cancel_flag):
                proc.terminate()
                try:
                    proc.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    proc.kill()
                set_status("CANCELLED", epoch=epoch, message="cancel.flag nakita")
                raise SystemExit("Cancelled.")

    code = proc.wait()
    if code != 0:
        set_status("FAILED", error=f"trainer exited with code {code}")
        raise SystemExit(f"FAILED (code {code})")
    print("training done")


## Cell 9 - `manifest.json` (pangalan, laki, sha256)

Ito ang binabasa ng `collectVerifiedArtifact()`. Kapag wala ito, o kapag hindi
tumugma ang sha256, hindi ipapasa ng app ang file sa ComfyUI.


In [ ]:
weights = sorted(glob.glob(os.path.join(output_dir, "*.safetensors")))
assert weights, f"Walang .safetensors sa {output_dir}"

files = [{"name": os.path.basename(w),
          "bytes": os.path.getsize(w),
          "sha256": sha256_file(w)} for w in weights]
print(json.dumps(files, indent=2))

if MODE == "job":
    write_json(os.path.join(job_dir(job_id), "manifest.json"), {"files": files})
    set_status("DONE", message=f"{len(files)} file(s)")
    print("Tapos. Hihintayin ng app ang DONE at i-verify ang sha256.")
else:
    print("Manual mode: hindi isinusulat ang manifest.json (walang job).")
